# IGLOO Playground — GB1 additive / Linear

An interactive, step-by-step companion to the production pipeline (`scripts/run_integrated_gradients.py`) for **GB1 additive, Linear interpolation** only.

Every heavy black-box call in the production script is unpacked here into separate cells so you can inspect **tensor shapes and raw values at each stage**: embeddings -> attention pooling -> integrated gradients -> the 7 evaluation metrics. Nothing here is a new algorithm — it's the same math as `src/igloo/sequence_ig.py`'s linear-IG branch (trapezoidal rule), just exposed for tuning and hypothesis-testing (e.g. "is the embedding space sparse enough that a different pooling/reduction technique would help?").

See `colab/README.md` for how to launch this with a GPU-backed kernel. Short version: kernel = the project's `IGLOO_env` venv, ideally on a compute node. Loading the trained checkpoint and reading precomputed CSVs is instant (existing disk artifacts); embedding a sequence for the first time and the integrated-gradients computation both benefit from a GPU but work on CPU too.

## 0. Setup

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

COLAB_DIR = Path.cwd() if (Path.cwd() / 'igloo_playground_utils.py').exists() else Path('.').resolve()
sys.path.insert(0, str(COLAB_DIR))
import igloo_playground_utils as ig

PROJECT_ROOT = ig.PROJECT_ROOT
DATA_DIR = COLAB_DIR / 'data'
print('colab dir     :', COLAB_DIR)
print('project root  :', PROJECT_ROOT)
print('device        :', ig.DEVICE)
print('cuda available:', torch.cuda.is_available())

## 1. Load the metrics summary and pick a run

`GB1_additive_signal_families_vs_model_perf_raw_linear.csv` has all 60 (noise, seed) cells x 7 metrics, already computed by `scripts/plot_ndcg_vs_model_performance.py`. First: which cell has the **best model performance** (Spearman rho between predicted and true fitness)? That's our primary focus per your request. We also show the best cell **per metric**, in case a metric other than model performance is what you want to dig into.

In [ ]:
metrics_df = pd.read_csv(DATA_DIR / 'GB1_additive_signal_families_vs_model_perf_raw_linear.csv')
baseline_df = pd.read_csv(DATA_DIR / 'GB1_additive_random_baseline_linear.csv')
print(f'{len(metrics_df)} (noise, seed) cells x {metrics_df.shape[1]} columns')
metrics_df.sort_values('x_value', ascending=False).head(8)

In [ ]:
METRIC_KEYS = ['ndcg', 'auroc', 'ap_graded', 'pv_ndcg_mean', 'sign_acc', 'cat_sep', 'mutation_ndcg_mean_per_seq']

print('=== best (noise, seed) per metric ===')
best_per_metric = []
for m in METRIC_KEYS:
    row = metrics_df.loc[metrics_df[m].idxmax()]
    best_per_metric.append({'metric': m, 'best_value': row[m], 'noise': row.noise, 'seed': row.seed, 'model_perf_x': row.x_value})
pd.DataFrame(best_per_metric)

### Selected run

Defaults to the single best model-performance cell. **Change these two lines to explore any other cell** (e.g. copy a (noise, seed) pair from the table above for a specific metric).

In [ ]:
best_row = metrics_df.loc[metrics_df['x_value'].idxmax()]
SELECTED_NOISE = best_row['noise']   # e.g. 0.0
SELECTED_SEED  = best_row['seed']    # e.g. 'seed_1'

# Match by numeric value against the noise_* directories on disk -- avoids float-formatting
# pitfalls (e.g. f'{0.0:g}' -> '0', but the directory is actually named 'noise_0.0').
gb1_additive_dir = PROJECT_ROOT / 'results' / 'synthetic' / 'GB1_additive'
noise_dirs = {float(p.name.split('_', 1)[1]): p for p in gb1_additive_dir.glob('noise_*') if p.is_dir()}
run_dir = noise_dirs[SELECTED_NOISE] / str(SELECTED_SEED)
linear_dir = run_dir / 'linear'
print('Selected run:', run_dir.relative_to(PROJECT_ROOT))
print(f"  model_test_spearman = {best_row['x_value']:.4f}")
for m in METRIC_KEYS:
    print(f'  {m:28s} = {best_row[m]:.4f}')

## 2. Load the trained model + config + official IG results

This just loads the existing checkpoint (no retraining) and the CSV the production pipeline already wrote for this cell — both instant.

In [ ]:
model, model_cfg, norm_params = ig.load_checkpoint(run_dir, dataset_name='GB1_additive')

In [ ]:
config_snapshots = sorted(linear_dir.glob('GB1_additive_config_snapshot_*.yaml'))
import yaml
with open(config_snapshots[-1]) as f:
    run_config = yaml.safe_load(f)
print(f'Config snapshot: {config_snapshots[-1].name}')
print(yaml.dump(run_config, sort_keys=False))

In [ ]:
official_csv = linear_dir / 'GB1_additive_integrated_gradients_results.csv'
official_df = pd.read_csv(official_csv)
print('shape:', official_df.shape)
print('columns:', list(official_df.columns))
official_df.head()

Cross-check: recompute the 7 metrics for this exact cell with `compute_metrics_for_dir` and confirm they match the summary table row above (sanity check that everything lines up).

In [ ]:
m = ig.compute_metrics_for_dir(linear_dir, protein='GB1', fitness_model='additive')
mutation_ndcg_mean = float(np.mean(m['mutation_ndcg_vals']))
print(f"{'metric':28s} {'summary CSV':>12s} {'recomputed':>12s}")
for key, recomputed in [('ndcg', m['ndcg']), ('auroc', m['auroc']), ('ap_graded', m['ap_graded']),
                        ('pv_ndcg_mean', m['pv_ndcg_mean']), ('sign_acc', m['sign_acc']),
                        ('cat_sep', m['cat_sep']), ('mutation_ndcg_mean_per_seq', mutation_ndcg_mean)]:
    print(f'{key:28s} {best_row[key]:12.4f} {recomputed:12.4f}')

## 3. Pick an example sequence and look at raw embeddings

First, some candidate sequences to inspect — the wildtype and a few real mutants from the test set, with their mutation count.

In [ ]:
dataset_df = pd.read_csv(DATA_DIR / 'GB1_additive.csv')
wt_seq = ig.load_wt_sequence('GB1')
print('WT sequence:', wt_seq)
print('WT length  :', len(wt_seq))

test_names = sorted(official_df[official_df['sequence_name'].str.contains(r'\[test\]', regex=True)]['sequence_name'].unique())
print(f'{len(test_names)} distinct test-split sequence entries, e.g.:')
test_names[:15]

**Pick a sequence to trace through the pipeline.** Change `SEQUENCE_NAME` to any value from the list above (or the WT itself). Defaults to the first non-WT test sequence.

In [ ]:
SEQUENCE_NAME = test_names[0]
variant_row = official_df[official_df['sequence_name'] == SEQUENCE_NAME].iloc[0]
print('Chosen sequence entry:', SEQUENCE_NAME)

# Reconstruct the full variant sequence from the official CSV's per-position mutation records
seq_rows = official_df[official_df['sequence_name'] == SEQUENCE_NAME].sort_values('position')
variant_seq = list(wt_seq)
for _, r in seq_rows.iterrows():
    variant_seq[int(r['position']) - 1] = r['amino_acid']
variant_seq = ''.join(variant_seq)
n_mut = sum(a != b for a, b in zip(wt_seq, variant_seq))
print(f'Variant sequence ({n_mut} mutation(s) vs WT):')
print(variant_seq)

### Step 3a — embed WT and the variant

Uses the same frozen ESM-C-300M backbone (layer 33) as the production pipeline. The first call below computes a fresh embedding (a few seconds on GPU, longer on CPU) and writes it to its own small cache file under `data/embeddings/synthetic/` (dataset_name=`GB1_additive_playground`) so re-running this cell, or asking for the same sequence again later, is then instant. **This is the tensor shape you'll see throughout: `(1, 56, 960)` = (batch, GB1 residues, ESM-C-300M embedding dim).**

In [ ]:
PLAYGROUND_DATASET_NAME = 'GB1_additive_playground'  # a small, notebook-local embedding cache
wt_emb = ig.get_sequence_embedding(wt_seq, dataset_name=PLAYGROUND_DATASET_NAME)
var_emb = ig.get_sequence_embedding(variant_seq, dataset_name=PLAYGROUND_DATASET_NAME)
print('wt_emb  shape:', wt_emb.shape, ' dtype:', wt_emb.dtype)
print('var_emb shape:', var_emb.shape, ' dtype:', var_emb.dtype)
print()
print(f'wt_emb  stats: mean={wt_emb.mean():.4f} std={wt_emb.std():.4f} min={wt_emb.min():.4f} max={wt_emb.max():.4f}')
print(f'var_emb stats: mean={var_emb.mean():.4f} std={var_emb.std():.4f} min={var_emb.min():.4f} max={var_emb.max():.4f}')
print()
print('Raw values, position 0 (first residue), first 8 embedding dims:')
print('  WT  :', wt_emb[0, 0, :8])
print('  var :', var_emb[0, 0, :8])

## 4. Hypothesis check: is the embedding space sparse?

If most of ESM-C's 960 embedding dimensions carry near-zero, redundant, or low-variance signal, that would argue for a different pooling/reduction technique than the model's plain attention-weighted average (e.g. a learned low-rank projection, or PCA-whitening before pooling). Let's check on a batch of real sequences rather than just one.

In [ ]:
# Bulk-embed a sample of sequences (one fresh ESM-C forward pass over 64 short sequences;
# a few seconds on GPU, a couple minutes on CPU -- bump N_SAMPLE up once you have a GPU kernel)
N_SAMPLE = 64
trainval_sample = dataset_df['SEQ'].astype(str).tolist()[:N_SAMPLE]
bulk_emb = ig.get_dataset_embeddings(
    trainval_sample, dataset_name='GB1_additive_playground_sparsity_sample', batch_size=64,
)
print('bulk_emb shape:', bulk_emb.shape, ' (N sequences, L positions, D dims)')

report = ig.embedding_sparsity_report(bulk_emb)
print(f"mean fraction near-zero per dim : {report['mean_frac_near_zero_per_dim']:.4f}")
print(f"max  fraction near-zero per dim : {report['max_frac_near_zero_per_dim']:.4f}")
print(f"dims needed for 50% variance    : {report['n_dims_for_50pct_variance']} / {report['d_embedding']}")
print(f"dims needed for 90% variance    : {report['n_dims_for_90pct_variance']} / {report['d_embedding']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.cumsum(report['explained_var_ratio'][:200]))
axes[0].axhline(0.9, color='grey', ls='--', lw=1)
axes[0].set_xlabel('# principal components'); axes[0].set_ylabel('cumulative explained variance')
axes[0].set_title('PCA of pooled-position embeddings')

axes[1].hist(report['frac_near_zero_per_dim'], bins=40, color='#4C72B0')
axes[1].set_xlabel('fraction of values near zero, per dimension'); axes[1].set_ylabel('# dims')
axes[1].set_title('Per-dimension sparsity')
fig.tight_layout(); plt.show()

**Interpretation space** — if the left plot saturates well before D=960 (e.g. 90% of variance in &lt;100 components), the model's attention-weighted average is working in an effectively low-dimensional space, and a PCA/learned-projection pooling head might recover the same signal with far fewer parameters (or expose which components actually drive predictions). If it's nearly a straight diagonal line, the space is closer to full-rank and this hypothesis is weaker.

## 5. Normalize, forward pass, and attention-pooling weights

Two things happen before the MLP: (1) the raw embedding is z-score normalised using the *training-set* mean/scale saved in the checkpoint, (2) the attention-pooling layer collapses `(1, 56, 960)` down to one 960-dim vector via a learned, position-wise softmax.

In [ ]:
mean, scale = norm_params['mean'], norm_params['scale']
var_emb_t = torch.tensor(var_emb, dtype=torch.float32, device=ig.DEVICE)
norm_emb = ig.normalize_embedding(var_emb_t, mean, scale)
print('normalized embedding shape:', tuple(norm_emb.shape))
print(f'post-normalization stats: mean={norm_emb.mean().item():.4f} std={norm_emb.std().item():.4f}')

pred = ig.model_forward(model, norm_emb, model_cfg['pooling_method'])
print(f'model prediction (rescaled fitness) shape: {tuple(pred.shape)}, value = {pred.item():.4f}')
print(f"actual fitness (from CSV)                 = {variant_row['actual_activity']:.4f}")

In [ ]:
attn_weights = ig.get_attention_pooling_weights(model, norm_emb)[0].cpu().numpy()  # (L,)
print('attention weights shape:', attn_weights.shape, ' sum =', attn_weights.sum())

iv, ann_mask, by_cat, bg_positions = ig.load_ground_truth('GB1')
fig, ax = plt.subplots(figsize=(11, 3.5))
colors = ['#E63946' if ann_mask[i] else '#AAB2BD' for i in range(len(attn_weights))]
ax.bar(range(1, len(attn_weights) + 1), attn_weights, color=colors)
ax.set_xlabel('residue position (1-indexed)'); ax.set_ylabel('attention weight')
ax.set_title(f'Attention-pooling weights for {SEQUENCE_NAME}  (red = biologically annotated position)')
fig.tight_layout(); plt.show()

## 6. Integrated gradients, step by step

This is the core of IGLOO: walk a linear path in embedding space from the WT baseline to the variant, accumulate gradients of the model's prediction with respect to the embedding at each step (trapezoidal rule), and scale by the total displacement. `steps=500` matches the production config exactly; try dropping it to see how attribution stability degrades with fewer steps.

In [ ]:
STEPS = 500       # production default; try e.g. 10, 50 to see the effect of coarser integration
BATCH_SIZE = 32

ig_result = ig.compute_linear_ig_stepwise(
    model, input_emb=var_emb, baseline_emb=wt_emb,
    mean=mean, scale=scale, pooling_method=model_cfg['pooling_method'],
    steps=STEPS, batch_size=BATCH_SIZE, verbose=True,
)
print()
print(f"completeness check (should be ~0): sum(signed_ig) - (F(var)-F(wt)) = {ig_result['completeness_gap']:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
colors = ['#E63946' if ann_mask[i] else '#AAB2BD' for i in range(len(ig_result['signed_ig']))]
ax.bar(range(1, len(ig_result['signed_ig']) + 1), ig_result['signed_ig'], color=colors)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('residue position'); ax.set_ylabel('signed integrated gradient')
ax.set_title(f'Per-position signed IG for {SEQUENCE_NAME}  ({STEPS} steps, linear path)')
fig.tight_layout(); plt.show()

### Cross-check against the official (production-computed) values
This reimplementation should closely match what's already in the results CSV for this exact sequence — small floating-point differences are expected, large ones would flag a bug.

In [ ]:
official_seq_df = ig.load_official_ig_for_sequence(official_csv, SEQUENCE_NAME)
mutated_positions = official_seq_df['position'].values.astype(int) - 1   # 0-indexed
official_vals = official_seq_df.sort_values('position')['integrated_gradient'].values
notebook_vals = ig_result['signed_ig'][mutated_positions]

cmp_df = pd.DataFrame({
    'position': official_seq_df.sort_values('position')['position'].values,
    'official_csv_ig': official_vals,
    'notebook_recomputed_ig': notebook_vals,
    'abs_diff': np.abs(official_vals - notebook_vals),
})
print(f"max abs diff: {cmp_df['abs_diff'].max():.6f}   Pearson r: {np.corrcoef(official_vals, notebook_vals)[0,1]:.6f}")
cmp_df

### Optional: gradient x input vs. full integrated gradients

The paper's headline finding #1 is that first-order gradient x input is often competitive with full path integration. Try it hands-on: `steps=1` collapses the trapezoidal integral to a single evaluation at the endpoint, i.e. exactly gradient x input. Compare its per-position ranking to the full `STEPS`-step result above.

In [ ]:
grad_x_input = ig.compute_linear_ig_stepwise(
    model, input_emb=var_emb, baseline_emb=wt_emb,
    mean=mean, scale=scale, pooling_method=model_cfg['pooling_method'],
    steps=1, batch_size=2, verbose=False,
)
corr = np.corrcoef(ig_result['signed_ig'], grad_x_input['signed_ig'])[0, 1]
print(f'Pearson correlation between full-IG ({STEPS} steps) and 1-step gradient x input: {corr:.4f}')

## 7. Recreate `GB1_additive_ndcg_vs_model_test_spearman_linear` inline

Same plot as `results/synthetic/figures/GB1_additive_ndcg_vs_model_test_spearman_linear.png`, rendered directly here (no file written). Change `Y_METRIC` to any of the 7 metrics to recreate any of the other family plots the same way.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.plot_ndcg_vs_model_performance import plot_one_interp, Y_METRIC_LABELS
%matplotlib inline
# ^ the import above sets matplotlib.use('Agg') when the module is run as a CLI
# script; harmless there, but it also fires on import, so we reassert the
# inline backend here or the figure below would silently fail to display.

Y_METRIC = 'ndcg'   # try: 'sign_acc', 'cat_sep', 'mutation_ndcg_mean_per_seq', 'auroc', 'ap_graded', 'pv_ndcg_mean'

plot_df = metrics_df.copy()
fig = plot_one_interp(
    plot_df, interp='linear', x_metric='model_test_spearman', y_metric=Y_METRIC,
    output_path=None, dpi=110, baseline_df=baseline_df, show=True,
)

## 8. Where to go from here

Things worth trying by editing cells above and re-running:
- **Pick a different run** in Section 1 (any (noise, seed) row) — note only the newly-generated decile-fill noise levels (0.2, 0.4, 0.6, 0.7, 0.9) currently have saved model checkpoints; the recovered noise=0.0/0.1/0.3/0.5/0.8/1.0 cells have IG results but need a checkpoint regenerated via `jobs/run_sweep_cell_gb1_deciles.sh` before Section 2 will find a `.pth` file (noise=0.0/seed_1, our default focus run, was regenerated for this notebook).
- **Pick a different sequence** in Section 3 — try a triple mutant vs. a single mutant, or a sequence with mutations at annotated vs. background positions.
- **Vary `STEPS`** in Section 6 to see how few integration steps you can get away with before attribution quality degrades (ties to the paper's finding that gradient x input is often competitive with full IG).
- **Extend `embedding_sparsity_report`** in Section 4 to more sequences, or compare its explained-variance curve across different noise levels / seeds to see if noise injection changes the embedding geometry the model actually uses.
- **Add a slerp or nearest-neighbor path**: `igloo_playground_utils.compute_linear_ig_stepwise` can be copied and adapted using `src.igloo.sequence_ig.interpolate_embeddings(..., method='slerp')` and `slerp_tangent(...)` for the tangent term, following the same pattern.